# Block 2: Local Structures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_2_2_cnn_advanced_blocks.ipynb)

## 2.2: CNNs: Advanced Blocks & Architectures

Short PyTorch demonstrations to accompany the slides: skip connections, parallel branches, upsampling, an encoder–decoder, and causal convolutions.

All examples run on CPU with synthetic inputs. Convolution weights are randomly initialized unless we explicitly set them; these examples illustrate operations and shapes, not learned predictions.

### 1. Beyond Basic CNNs

In 2.1, we followed a sequence of convolutions and pooling layers to obtain class logits. Here we introduce paths that **branch and merge**, networks that produce **spatial outputs**, and convolutions that respect **temporal order**.

**Before running:** predict the shapes requested in each section. Each shape example starts with `show_shapes = False`: input shapes are always shown, while other shapes appear as `shape?`. Change the flag to `True` in that example and rerun the cell to reveal its shapes. Each example has its own flag, so revealing one example does not reveal the next. The plots and small numerical examples appear when you run their cells.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=2, sci_mode=False)

show_shapes = False  # Set True after making your predictions.


def print_shapes(**tensors):
    for name, tensor in tensors.items():
        shape = tuple(tensor.shape) if show_shapes or name == "input" else "shape?"
        print(f"{name}: {shape}")
    return show_shapes

### 2. Skip Connections

#### Residual Blocks: Add the Input Back

A residual block combines a learned transformation with a shortcut:

$$y = x + F(x).$$

Here $F$ contains two 3×3 convolutions with a ReLU between them. Padding preserves spatial size, and both convolutions preserve channels. We focus on the residual sum; this teaching example omits the batch normalization and final activation used in a standard ResNet basic block.

**Predict before running:** What are the shapes of $F(x)$ and $y$? If $F(x)$ were zero, what would the block return?

In [ ]:
show_shapes = False  # Change to True after making your predictions.

# Shape: (N, C, H, W) = (batch size, channels, height, width).
x = torch.randn(1, 4, 8, 8)  # One sample, four channels, an 8×8 grid.
residual_branch = nn.Sequential(
    nn.Conv2d(in_channels=4, out_channels=4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(in_channels=4, out_channels=4, kernel_size=3, padding=1),
)

with torch.inference_mode():
    fx = residual_branch(x)
    residual_output = x + fx

print_shapes(input=x, residual_branch=fx, output=residual_output);

Addition combines corresponding values and preserves the number of channels. Residual addition requires matching shapes: if a ResNet block changes channels or resolution, a projection shortcut (typically a 1×1 convolution, with stride when needed) aligns the input with the branch output.

#### DenseNet: Concatenate and Reuse Features

Concatenation keeps feature maps as separate channels. In a DenseNet block, each layer receives the original input and all earlier layers' outputs. Each layer adds a fixed number of new channels, called the **growth rate**.

Below, each layer adds two channels. `dim=1` selects the channel dimension in `(N, C, H, W)`.

**Predict before running:** Starting with four channels, how many channels enter the second layer? How many are present after both concatenations? Could we replace the first concatenation with addition here?

In [ ]:
show_shapes = False  # Change to True after making your predictions.

dense_layer1 = nn.Sequential(
    nn.Conv2d(in_channels=4, out_channels=2, kernel_size=3, padding=1),
    nn.ReLU(),
)
dense_layer2 = nn.Sequential(
    nn.Conv2d(in_channels=6, out_channels=2, kernel_size=3, padding=1),
    nn.ReLU(),
)

with torch.inference_mode():
    new_features1 = dense_layer1(x)
    features1 = torch.cat([x, new_features1], dim=1)
    new_features2 = dense_layer2(features1)
    features2 = torch.cat([features1, new_features2], dim=1)

print_shapes(input=x, new_features1=new_features1,
             after_layer1=features1, new_features2=new_features2,
             after_layer2=features2);

### 3. Multi-scale Feature Extraction: An Inception Module

GoogLeNet uses Inception modules to process the same input through parallel branches and concatenate their outputs. Different kernel sizes use different spatial neighbourhoods.

This small Inception-style module has four branches. The 1×1 convolutions before the 3×3 and 5×5 convolutions reduce channels, making those spatial convolutions cheaper. The pooling branch uses stride 1 and padding to preserve resolution.

**Predict before running:** For input `(1, 4, 8, 8)`, what does each branch output? How many channels does the merged tensor have? Why do all branches need the same height and width?

In [ ]:
show_shapes = False  # Change to True after making your predictions.

branches = nn.ModuleDict({
    "1x1": nn.Sequential(
        nn.Conv2d(in_channels=4, out_channels=2, kernel_size=1),
        nn.ReLU(),
    ),
    "3x3": nn.Sequential(
        nn.Conv2d(in_channels=4, out_channels=2, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=2, out_channels=4, kernel_size=3, padding=1),
        nn.ReLU(),
    ),
    "5x5": nn.Sequential(
        nn.Conv2d(in_channels=4, out_channels=2, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=2, out_channels=4, kernel_size=5, padding=2),
        nn.ReLU(),
    ),
    "pool": nn.Sequential(
        nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
        nn.Conv2d(in_channels=4, out_channels=2, kernel_size=1),
        nn.ReLU(),
    ),
})

with torch.inference_mode():
    branch_outputs = {name: branch(x) for name, branch in branches.items()}
    inception_output = torch.cat(list(branch_outputs.values()), dim=1)

print_shapes(input=x, **branch_outputs, merged=inception_output);

### 4. Upsampling and Encoder–Decoder Networks

#### Interpolation: Increase Spatial Resolution

[`F.interpolate`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html) resizes each channel independently. **Nearest neighbour** copies nearby values; **bilinear interpolation** mixes nearby values. Neither has learnable weights.

**Predict before running:** What is the output shape when we double this 2×2 input? Which method introduces intermediate values?

In [ ]:
show_shapes = False  # Change to True after making your predictions.

# Add batch and channel dimensions: (N, C, H, W) = (1, 1, 2, 2).
small_grid = torch.tensor([[0., 1.], [1., 0.]]).reshape(1, 1, 2, 2)
nearest = F.interpolate(small_grid, scale_factor=2, mode="nearest")
bilinear = F.interpolate(small_grid, scale_factor=2, mode="bilinear",
                         align_corners=False)

print_shapes(input=small_grid, nearest=nearest, bilinear=bilinear)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, title, grid in zip(axes, ["Input", "Nearest", "Bilinear"],
                           [small_grid, nearest, bilinear]):
    ax.imshow(grid[0, 0], cmap="gray", vmin=0, vmax=1,
              interpolation="nearest")
    ax.set_title(title)
    height, width = grid.shape[-2:]
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticks([i - 0.5 for i in range(width + 1)], minor=True)
    ax.set_yticks([i - 0.5 for i in range(height + 1)], minor=True)
    ax.grid(which="minor", color="tab:blue", linewidth=1)
    ax.tick_params(which="minor", bottom=False, left=False)
plt.tight_layout()
plt.show()

#### Transposed Convolution: Learnable Upsampling

[`nn.ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html) can increase spatial resolution using learnable kernel weights. It does not generally invert a convolution.

Here a 2×2 kernel with stride 2 places each input value into a separate 2×2 output patch. We set all kernel weights to one to make the operation visible.

**Predict before running:** What will the output grid contain? Which interpolation result above will it match with these weights?

In [ ]:
show_shapes = False  # Change to True after making your predictions.

upsample = nn.ConvTranspose2d(in_channels=1, out_channels=1,
                              kernel_size=2, stride=2, bias=False)
with torch.no_grad():
    upsample.weight.fill_(1)
    transposed_output = upsample(small_grid)

print("Kernel:\n", upsample.weight[0, 0].detach())
if print_shapes(output=transposed_output):
    print("Output grid:\n", transposed_output[0, 0])


#### U-Net: Combine Context and Spatial Detail

An encoder reduces spatial resolution while building features. A decoder upsamples those features. U-Net also passes encoder features directly to the decoder at matching resolutions, providing spatial detail alongside deeper context.

This **one-level U-Net-style example** uses bilinear upsampling and channel concatenation. Its output contains three class logits **at each pixel**. The original U-Net has multiple levels and more convolutions per level.

**Predict before running:** Track the shapes from `(1, 1, 16, 16)`. How many channels enter the decoder after concatenation? How does the final output differ from the classification logits in 2.1?

In [ ]:
show_shapes = False  # Change to True after making your predictions.

# Shape: (N, C, H, W); one image, one channel, 16×16 pixels.
image = torch.randn(1, 1, 16, 16)
encoder = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1),
    nn.ReLU(),
)
pool = nn.MaxPool2d(kernel_size=2) # by default, stride = kernel_size
bottleneck = nn.Sequential(
    nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, padding=1),
    nn.ReLU(),
)
decoder = nn.Sequential(
    nn.Conv2d(in_channels=12, out_channels=4, kernel_size=3, padding=1),
    nn.ReLU(),
)
head = nn.Conv2d(in_channels=4, out_channels=3, kernel_size=1)

with torch.inference_mode():
    skip = encoder(image)
    down = pool(skip)
    context = bottleneck(down)
    up = F.interpolate(context, size=skip.shape[-2:], mode="bilinear",
                       align_corners=False)
    merged = torch.cat([skip, up], dim=1)
    segmentation_logits = head(decoder(merged))

print_shapes(input=image, encoder_skip=skip, downsampled=down,
             context=context, upsampled=up, concatenated=merged,
             segmentation_logits=segmentation_logits);

### 5. CNNs for Audio Generation

#### Causal Convolutions: An Impulse Response

A 1D signal has shape **`(N, C, L)`**. A causal convolution's output at time $t$ uses only input positions at or before $t$. We achieve this with **left-only zero padding**. Dilation spaces the kernel taps further apart, increasing temporal coverage without adding weights or reducing output length.

We use a three-tap averaging kernel with no bias. For dilation $d$, its taps read $x_t$, $x_{t-d}$ and $x_{t-2d}$.

**Predict before running:** An impulse occurs at $t=8$. Mark the output positions that can be nonzero for dilations 1, 2 and 4. Can the output respond before the impulse?

Then change `signal_kind` to `"step"`: how does the transition spread over time?

In [ ]:
signal_kind = "impulse"  # Change to "step" and rerun.
length, onset = 24, 8
# Shape: (N, C, L) = (batch size, channels, sequence length).
signal = torch.zeros(1, 1, length)
if signal_kind == "impulse":
    signal[..., onset] = 1
elif signal_kind == "step":
    signal[..., onset:] = 1
else:
    raise ValueError('Choose "impulse" or "step".')

fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharex=True, sharey=True)
axes[0].stem(range(length), signal[0, 0].numpy(), basefmt=" ")
axes[0].set_title(f"Input: {signal_kind}")

for ax, dilation in zip(axes[1:], [1, 2, 4]):
    conv = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3,
                     dilation=dilation, bias=False)
    left_padding = (conv.kernel_size[0] - 1) * dilation
    with torch.no_grad():
        conv.weight.fill_(1 / 3)
        response = conv(F.pad(signal, (left_padding, 0)))
    ax.stem(range(length), response[0, 0].numpy(), basefmt=" ")
    ax.set_title(f"Causal convolution: dilation {dilation}")

for ax in axes:
    ax.axvline(onset, color="gray", linestyle="--", linewidth=1)
    ax.set_ylim(-0.05, 1.1)
    ax.set_ylabel("Value")
    ax.grid(alpha=0.2)
axes[-1].set_xticks(range(0, length, 2))
axes[-1].set_xlabel("Time index")
plt.tight_layout()
plt.show()

#### [optional] Stacking Causal Layers: Reach Further into the Past

Here the layers run **in sequence**: each receives the previous layer's output. With a three-tap kernel and dilation 2, each layer extends the temporal context by four positions. After four layers, an output can depend on inputs as far as sixteen time steps earlier. With fixed dilation 2, only every second input position is reached.

**Predict before running:** For an impulse at $t=8$, how late can a nonzero output appear after one, two, three and four layers?

Then change `dilations` to `[1, 2, 4, 8]` to try the doubling pattern used in [WaveNet](https://arxiv.org/abs/1609.03499). How does the reach change? Do gaps remain in the final response?

In [ ]:
dilations = [2, 2, 2, 2]  # Then try [1, 2, 4, 8].
length, onset = 40, 8
# Shape: (N, C, L) = (batch size, channels, sequence length).
impulse = torch.zeros(1, 1, length)
impulse[..., onset] = 1

# Each stage pads on the left, then applies a three-tap convolution.
stack = nn.Sequential(*[
    nn.Sequential(
        nn.ZeroPad1d(padding=(2 * dilation, 0)),
        nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3,
                  dilation=dilation, bias=False),
    )
    for dilation in dilations
])
with torch.no_grad():
    for stage in stack:
        stage[1].weight.fill_(1 / 3)

fig, axes = plt.subplots(len(stack) + 1, 1, figsize=(9, 6),
                         sharex=True, sharey=True)
axes[0].stem(range(length), impulse[0, 0].numpy(), basefmt=" ")
axes[0].set_title("Input: impulse")
h = impulse
for depth, (stage, ax) in enumerate(zip(stack, axes[1:]), start=1):
    with torch.no_grad():
        h = stage(h)
    reach = 2 * sum(dilations[:depth])
    ax.stem(range(length), h[0, 0].numpy(), basefmt=" ")
    ax.axvspan(onset, onset + reach, color="tab:blue", alpha=0.08)
    ax.set_title(f"After {depth} layer(s): reaches {reach} time steps back")
    print(f"After {depth} layer(s), the impulse influences outputs through t={onset + reach}.")

for ax in axes:
    ax.axvline(onset, color="gray", linestyle="--", linewidth=1)
    ax.set_ylim(-0.05, 1.1)
    ax.set_ylabel("Value")
    ax.grid(alpha=0.2)
axes[-1].set_xticks(range(0, length, 2))
axes[-1].set_xlabel("Time index")
plt.tight_layout()
plt.show()